# Lens correction: Kaggle data → Preprocessing → FlowWarpNet

**Single notebook:** data from Kaggle, then preprocessing (Step 0), then FlowWarpNet training.

---

## Initial setup (edit below)

1. **Runtime → Change runtime type → Hardware accelerator: GPU**
2. Set your **Kaggle dataset** in the next cell (e.g. `username/dataset-name`).
3. Add your Kaggle API key: upload `kaggle.json` when prompted, or use *Secrets* in Colab.

Data will be downloaded from Kaggle (no desktop upload). Output: preprocessed datasets + `checkpoints/best.pt` (saved to Drive if mounted).

## 1. Kaggle — download your dataset

In [ ]:
# Set your Kaggle dataset (e.g. from dataset URL: kaggle.com/datasets/USERNAME/DATASET)
KAGGLE_DATASET = "joshcompanion/lens-correction-train"  # ← change to your dataset
DOWNLOAD_DIR = "/content/kaggle_data"
DATA_DIR = "/content/data"

!pip install -q kaggle
import os
from pathlib import Path

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Kaggle API: upload kaggle.json to Colab (Files panel) or add to Secrets
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    from google.colab import files
    print("Upload your kaggle.json (Kaggle → Account → Create New API Token)")
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d "{KAGGLE_DATASET}" -p "{DOWNLOAD_DIR}"

# Unzip (common names)
zips = list(Path(DOWNLOAD_DIR).glob("*.zip"))
if zips:
    import zipfile
    for z in zips:
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(DOWNLOAD_DIR)
    print("Extracted:", [str(z) for z in zips])

# Find training_metadata.json and set SOURCE_DIR
def find_metadata(root):
    root = Path(root)
    for p in root.rglob("training_metadata.json"):
        return p.parent
    return None

SOURCE_DIR = find_metadata(DOWNLOAD_DIR)
if SOURCE_DIR is None:
    SOURCE_DIR = Path(DOWNLOAD_DIR)
METADATA_PATH = Path(SOURCE_DIR) / "training_metadata.json"
if not METADATA_PATH.exists():
    METADATA_PATH = None
print("SOURCE_DIR:", SOURCE_DIR)
print("METADATA_PATH:", METADATA_PATH)

## 2. Mount Drive and install dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# PyTorch is usually pre-installed on Colab
# !pip install torch torchvision Pillow

## 3. Get project code

In [ ]:
# Option A: Clone from GitHub (replace with your repo)
# !git clone https://github.com/YOUR_USERNAME/imagedistortion.git
# %cd imagedistortion

# Option B: Use project from Google Drive
%cd /content/drive/MyDrive/imagedistortion

## 4. Data preprocessing (Step 0)

Build raw_dataset, dataset_a (originals + corrected), dataset_b (by lens), and manifest from Kaggle data.

In [ ]:
import csv, json, re, shutil
from pathlib import Path
from typing import Any, Optional

def _sanitize_lens_folder_name(lens_model: Optional[str], camera_make: Optional[str]) -> str:
    if not lens_model or not str(lens_model).strip():
        return "others"
    parts = [p for p in (camera_make, lens_model) if p and str(p).strip()]
    raw = "_".join(str(p).strip() for p in parts) if parts else str(lens_model).strip()
    safe = re.sub(r"[^\w\-]", "_", raw)
    safe = re.sub(r"_+", "_", safe).strip("_")
    return safe or "others"

def load_metadata(metadata_path: Path) -> list:
    with open(metadata_path, encoding="utf-8") as f:
        data = json.load(f)
    pairs = data.get("pairs") or data.get("pair_ids") or []
    return pairs if isinstance(pairs, list) else list(pairs.values()) if isinstance(pairs, dict) else []

def build_raw_dataset(source_dir: Path, output_base: Path, use_symlinks: bool = False) -> dict:
    output_base, source_dir = output_base.resolve(), source_dir.resolve()
    if output_base == source_dir:
        return {"path": str(output_base), "message": "Same as source; no copy.", "files_copied": 0}
    if not source_dir.exists():
        return {"path": str(output_base), "message": "Source missing.", "files_copied": 0}
    if output_base.exists():
        shutil.rmtree(output_base)
    shutil.copytree(source_dir, output_base, symlinks=use_symlinks, ignore_dangling_symlinks=True, dirs_exist_ok=False)
    n = sum(1 for _ in output_base.rglob("*") if _.is_file())
    return {"path": str(output_base), "source": str(source_dir), "files_copied": n, "message": f"Raw dataset: {n} files."}

def build_dataset_a(metadata_path: Path, source_dir: Path, output_base: Path, use_symlinks: bool = False) -> dict:
    originals_dir = output_base / "originals"
    corrected_dir = output_base / "corrected"
    originals_dir.mkdir(parents=True, exist_ok=True)
    corrected_dir.mkdir(parents=True, exist_ok=True)
    pairs = load_metadata(metadata_path)
    copied = 0
    for pair in pairs:
        pair_id = pair.get("pair_id") or pair.get("id") or ""
        orig_name, gen_name = pair.get("original") or "", pair.get("generated") or ""
        if not pair_id or not orig_name or not gen_name: continue
        src_orig, src_gen = source_dir / orig_name, source_dir / gen_name
        if not src_orig.exists() or not src_gen.exists(): continue
        safe_id = re.sub(r"[^\w\-]", "_", pair_id)[:120]
        dest_orig = originals_dir / f"{safe_id}_original{Path(orig_name).suffix}"
        dest_corrected = corrected_dir / f"{safe_id}_corrected{Path(gen_name).suffix}"
        try:
            if use_symlinks:
                if not dest_orig.exists(): dest_orig.symlink_to(src_orig.resolve())
                if not dest_corrected.exists(): dest_corrected.symlink_to(src_gen.resolve())
            else:
                shutil.copy2(src_orig, dest_orig)
                shutil.copy2(src_gen, dest_corrected)
            copied += 1
        except OSError:
            pass
    return {"path": str(output_base.resolve()), "copied_pairs": copied, "total_pairs_in_metadata": len(pairs)}

def build_dataset_b(metadata_path: Path, source_dir: Path, output_base: Path, use_symlinks: bool = False) -> dict:
    pairs = load_metadata(metadata_path)
    lens_counts = {}
    copied = 0
    for pair in pairs:
        pair_id = pair.get("pair_id") or pair.get("id") or ""
        orig_name, gen_name = pair.get("original") or "", pair.get("generated") or ""
        if not pair_id or not orig_name or not gen_name: continue
        lens_id = _sanitize_lens_folder_name(pair.get("lens_model"), pair.get("camera_make"))
        lens_dir = output_base / lens_id
        lens_dir.mkdir(parents=True, exist_ok=True)
        lens_counts[lens_id] = lens_counts.get(lens_id, 0) + 1
        src_orig, src_gen = source_dir / orig_name, source_dir / gen_name
        if not src_orig.exists() or not src_gen.exists(): continue
        safe_id = re.sub(r"[^\w\-]", "_", pair_id)[:120]
        dest_orig = lens_dir / f"{safe_id}_original{Path(orig_name).suffix}"
        dest_corrected = lens_dir / f"{safe_id}_corrected{Path(gen_name).suffix}"
        try:
            if use_symlinks:
                if not dest_orig.exists(): dest_orig.symlink_to(src_orig.resolve())
                if not dest_corrected.exists(): dest_corrected.symlink_to(src_gen.resolve())
            else:
                shutil.copy2(src_orig, dest_orig)
                shutil.copy2(src_gen, dest_corrected)
            copied += 1
        except OSError:
            pass
    return {"path": str(output_base.resolve()), "lens_folders": list(lens_counts.keys()), "lens_counts": lens_counts, "others_count": lens_counts.get("others", 0), "total_pairs_copied": copied}

def generate_manifest(metadata_path: Path, output_base: Path) -> dict:
    pairs = load_metadata(metadata_path)
    lens_counts = {}
    for pair in pairs:
        lens_id = _sanitize_lens_folder_name(pair.get("lens_model"), pair.get("camera_make"))
        lens_counts[lens_id] = lens_counts.get(lens_id, 0) + 1
    total = sum(lens_counts.values())
    out_dir = Path(output_base)
    out_dir.mkdir(parents=True, exist_ok=True)
    json_path = out_dir / "manifest.json"
    csv_path = out_dir / "manifest.csv"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"total_pairs": total, "others_count": lens_counts.get("others", 0), "num_lens_types": len(lens_counts), "lenses": [{"lens_id": k, "count": v} for k, v in sorted(lens_counts.items(), key=lambda x: (-x[1], x[0]))], "lens_counts": lens_counts}, f, indent=2)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["lens_id", "count"])
        for lid, c in sorted(lens_counts.items(), key=lambda x: (-x[1], x[0])):
            w.writerow([lid, c])
        w.writerow(["others_count", lens_counts.get("others", 0)])
        w.writerow(["total_pairs", total])
    return {"manifest_json": str(json_path.resolve()), "total_pairs": total, "num_lens_types": len(lens_counts)}

# Run Step 0 using Kaggle SOURCE_DIR and METADATA_PATH
source = Path(SOURCE_DIR)
data_dir = Path(DATA_DIR)
data_dir.mkdir(parents=True, exist_ok=True)
use_symlinks = True

if METADATA_PATH is None or not Path(METADATA_PATH).exists():
    raise FileNotFoundError(f"training_metadata.json not found under {DOWNLOAD_DIR}. Check Kaggle dataset structure.")

meta = Path(METADATA_PATH)
r0 = build_raw_dataset(source, data_dir / "raw_dataset", use_symlinks=use_symlinks)
print("[Raw dataset]", r0.get("message", ""))

r1 = build_dataset_a(meta, source, data_dir / "dataset_a", use_symlinks=use_symlinks)
print("[Dataset A]", r1["copied_pairs"], "pairs")

r2 = build_dataset_b(meta, source, data_dir / "dataset_b", use_symlinks=use_symlinks)
print("[Dataset B]", len(r2["lens_folders"]), "lens folders")

r3 = generate_manifest(meta, data_dir)
print("[Manifest]", r3["num_lens_types"], "lens types, total_pairs:", r3["total_pairs"])
print("Preprocessing done. Output:", DATA_DIR)

## 5. Prepare flat train directory for FlowWarpNet

FlowWarpNet expects `*_Original.jpg` and `*_generated.jpg` in one folder. We build that from dataset_a.

In [ ]:
TRAIN_FLAT = "/content/data/train_flat"
Path(TRAIN_FLAT).mkdir(parents=True, exist_ok=True)

originals_dir = Path(DATA_DIR) / "dataset_a" / "originals"
corrected_dir = Path(DATA_DIR) / "dataset_a" / "corrected"

for orig in originals_dir.iterdir():
    if not orig.is_file(): continue
    base = orig.stem.replace("_original", "").strip("_")
    corr = corrected_dir / f"{base}_corrected{orig.suffix}"
    if not corr.exists(): continue
    shutil.copy2(orig, Path(TRAIN_FLAT) / f"{base}_Original{orig.suffix}")
    shutil.copy2(corr, Path(TRAIN_FLAT) / f"{base}_generated{orig.suffix}")

n_pairs = len(list(Path(TRAIN_FLAT).glob("*_Original*")))
TRAIN_DIR = TRAIN_FLAT
print(f"Train flat dir: {TRAIN_DIR} ({n_pairs} pairs)")

## 6. Train FlowWarpNet

In [ ]:
import subprocess

subprocess.run([
    "python", "-m", "lens_correction.train",
    "--train_dir", TRAIN_DIR,
    "--flat", "--subset", "5000", "--epochs", "1",
    "--batch_size", "8", "--lr", "2e-4", "--num_workers", "2",
    "--checkpoint_dir", "./checkpoints", "--save_best", "best.pt",
], check=True)

## 7. Save checkpoint to Drive

In [ ]:
import shutil

drive_ckpt_dir = "/content/drive/MyDrive/imagedistortion_colab_checkpoints"
Path(drive_ckpt_dir).mkdir(parents=True, exist_ok=True)
shutil.copy("checkpoints/best.pt", f"{drive_ckpt_dir}/best.pt")
print("Saved best.pt to Drive:", drive_ckpt_dir)

## 8. (Optional) Run inference and download zip

In [ ]:
TEST_DIR = "/content/drive/MyDrive/Dataset-Test"  # or path to test images on Drive
if os.path.isdir(TEST_DIR):
    subprocess.run([
        "python", "-m", "lens_correction.infer",
        "--test_dir", TEST_DIR, "--output_dir", "./outputs",
        "--checkpoint", "./checkpoints/best.pt", "--zip_path", "./submission_images.zip",
    ], check=True)
    from google.colab import files
    files.download("submission_images.zip")
else:
    print("Skipping inference. Set TEST_DIR to a folder with test images if needed.")